# Logistic Regression with scikit-learn

Follow the standard supervised-learning workflow: prepare labeled data, split it into training and test sets, fit only on training data, and evaluate predictions on the held-out test set. This example uses a small synthetic student dataset; it is illustrative, not real student data.


## 1. Create a labeled dataset

The inputs are study hours and attendance percentage; the target is `passed` (`1` = pass, `0` = fail). In a real project, load a documented dataset and check its feature definitions and target labels before modeling.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


rng = np.random.default_rng(42)
n_students = 200
hours_studied = rng.uniform(0, 10, n_students)
attendance_pct = rng.uniform(40, 100, n_students)

# A noisy rule creates an example target for teaching.
score = (
    0.65 * hours_studied
    + 0.035 * attendance_pct
    + rng.normal(0, 0.8, n_students)
)
students = pd.DataFrame(
    {
        "hours_studied": hours_studied,
        "attendance_pct": attendance_pct,
        "passed": (score >= 6.5).astype(int),
    }
)

students.head()


## 2. Split the data and fit the model

Keep a stratified test split for final evaluation and set aside part of the training data as validation data to compare thresholds. Standardizing the features helps the optimizer because hours and attendance use different units; the scaler is fitted inside the pipeline on fitting data only.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


features = ["hours_studied", "attendance_pct"]
X = students[features]
y = students["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train,
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X_fit, y_fit)

print("Fitting rows:", len(X_fit))
print("Validation rows:", len(X_validation))
print("Test rows:", len(X_test))
print("Class labels:", model.classes_)


## 3. Inspect probabilities and coefficients

`predict_proba` returns one probability per class, in the order shown by `classes_`. The standardized-feature coefficient describes the change in log-odds for a one-standard-deviation increase in that feature, holding the other feature fixed. Its exponent is the corresponding odds multiplier; it is not a causal effect.


In [ ]:
classifier = model.named_steps["logisticregression"]
coefficient_table = pd.DataFrame(
    {
        "coefficient": classifier.coef_[0],
        "odds_multiplier_per_standard_deviation": np.exp(classifier.coef_[0]),
    },
    index=features,
)

coefficient_table


In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]
y_predicted = (y_probability >= 0.5).astype(int)

print("First five P(pass):", y_probability[:5].round(3))
print("First five predictions:", y_predicted[:5])


## 4. Evaluate with a confusion matrix and classification metrics

A confusion matrix uses actual classes for rows and predicted classes for columns. With labels `[0, 1]`, its cells are true negatives, false positives, false negatives, and true positives, in that order. A **false positive** is a false alarm (predicted 1, actual 0); a **false negative** is a miss (predicted 0, actual 1).

Precision is $TP/(TP+FP)$ and helps avoid false alarms. Recall is $TP/(TP+FN)$ and helps avoid misses. F1 is the harmonic mean of precision and recall, $2PR/(P+R)$. Accuracy is $(TP+TN)/\text{all examples}$; it can hide poor performance on a class of interest, especially when classes are imbalanced.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


print("Confusion matrix (rows: actual 0/1, columns: predicted 0/1):")
print(confusion_matrix(y_test, y_predicted, labels=[0, 1]))
print(f"Accuracy:  {accuracy_score(y_test, y_predicted):.3f}")
print(f"Precision: {precision_score(y_test, y_predicted, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_predicted, zero_division=0):.3f}")
print(f"F1 score:  {f1_score(y_test, y_predicted, zero_division=0):.3f}")


## 5. Compare decision thresholds

Use validation probabilities to see how thresholds change predictions. Lowering the threshold usually increases recall and may reduce precision; the exact result depends on the data. Select a threshold using validation data or cross-validation, then evaluate the chosen rule once on the untouched test set.


In [ ]:
validation_probability = model.predict_proba(X_validation)[:, 1]
threshold_results = []
for threshold in [0.4, 0.5, 0.6]:
    threshold_predictions = (validation_probability >= threshold).astype(int)
    threshold_results.append(
        {
            "threshold": threshold,
            "precision": precision_score(
                y_validation, threshold_predictions, zero_division=0
            ),
            "recall": recall_score(
                y_validation, threshold_predictions, zero_division=0
            ),
            "f1": f1_score(y_validation, threshold_predictions, zero_division=0),
        }
    )

pd.DataFrame(threshold_results).set_index("threshold").round(3)


## 6. Predict for one new example

For a new row, provide the same feature names and units used during training. The returned probabilities correspond to `model.classes_`; here class 0 is fail and class 1 is pass.


In [ ]:
new_student = pd.DataFrame(
    [[6, 80]],
    columns=features,
)

pd.Series(
    model.predict_proba(new_student)[0],
    index=model.classes_,
    name="probability",
)
